# 13_RAG_Knowledge_Base

Builds the retrieval knowledge base used by the Career Advisor.

### Steps
1. Assemble rich skill documents from all pipeline outputs
2. Create `skill_knowledge_base.json` — one structured doc per skill
3. Build a FAISS vector index for semantic retrieval
4. Save `faiss_index.bin` + `faiss_metadata.json`

### Outputs
- `skill_knowledge_base.json`
- `faiss_index.bin`
- `faiss_metadata.json`


In [1]:
import pandas as pd
import numpy as np
import json
import os
import warnings
warnings.filterwarnings('ignore')

# ── load all pipeline outputs ─────────────────────────────────────────────────
ses       = pd.read_csv('../Generated Datasets/ses_rankings.csv')
forecasts = pd.read_csv('../Generated Datasets/forecast_results.csv')
history   = pd.read_csv('../Generated Datasets/skill_demand_history_clean.csv')
industry  = pd.read_csv('../Generated Datasets/industry_analytics.csv')
ind_map   = pd.read_csv('../Generated Datasets/industry_skill_map.csv')

for df in [ses, forecasts, history, ind_map]:
    df['skill'] = df['skill'].str.lower().str.strip()

print("SES       :", ses.shape)
print("Forecasts :", forecasts.shape)
print("History   :", history.shape)
print("Ind map   :", ind_map.shape)


SES       : (114, 23)
Forecasts : (114, 12)
History   : (2304, 4)
Ind map   : (139, 8)


In [2]:
# ── build industry membership per skill ──────────────────────────────────────
skill_industries = (
    ind_map.groupby('skill')['industry']
    .apply(list)
    .reset_index()
    .rename(columns={'industry': 'industries'})
)

# ── historical demand series per skill ───────────────────────────────────────
def demand_series(skill_name):
    sub = history[history['skill'] == skill_name].sort_values('year')
    return {str(int(row['year'])): int(row['demand'])
            for _, row in sub.iterrows()}

# ── merge all features ────────────────────────────────────────────────────────
master = ses.merge(
    forecasts[['skill','forecast_1y','forecast_2y','forecast_3y',
               'forecast_trend','forecast_method','confidence','history_length']],
    on='skill', how='left'
).merge(skill_industries, on='skill', how='left')

master['industries'] = master['industries'].apply(
    lambda x: x if isinstance(x, list) else []
)

print("Master frame:", master.shape)
master.head(3)


Master frame: (114, 31)


,ses_rank,skill,sub_category,ses_score,ses_tier,feature_score,forecast_intelligence,cluster_score,archetype,cluster,...,confidence_x,history_length_x,forecast_1y_y,forecast_2y_y,forecast_3y_y,forecast_trend_y,forecast_method_y,confidence_y,history_length_y,industries
0,1,javascript,Programming,0.767596,Very Safe,0.547114,0.925000,0.9,Future-Proof,3.0,...,0.85,13,279247.8901,292893.7543,306270.9869,Stable,Holt,0.85,13,[Software Engineering]
1,2,python,Programming,0.720562,Very Safe,0.615255,0.712743,0.9,Future-Proof,3.0,...,0.85,13,217685.3807,220769.2226,223311.0592,Stable,Holt,0.85,13,"[Data Science, Software Engineering, Data Anal..."
2,3,sql,Database,0.684792,Very Safe,0.536122,0.700982,0.9,Future-Proof,3.0,...,0.85,13,208242.1689,216057.9592,223507.8726,Stable,Holt,0.85,13,"[Data Engineering, Data Analytics]"


In [3]:
# ── build knowledge base documents ───────────────────────────────────────────
def build_document(row):
    skill = row['skill']
    tier  = row.get('ses_tier', 'Unknown')
    score = round(float(row.get('ses_score', 0)), 4)

    # Tier-aware narrative
    tier_narrative = {
        'Very Safe'      : f"{skill} is a highly future-proof skill with strong and growing demand.",
        'Safe'           : f"{skill} is a stable skill with healthy market demand.",
        'Moderate'       : f"{skill} shows moderate market relevance with mixed growth signals.",
        'Risky'          : f"{skill} faces declining demand and may require reskilling.",
        'Extinction Risk': f"{skill} is at high risk of obsolescence in the near term.",
    }.get(tier, f"{skill} has been assessed for workforce relevance.")

    # Forecast narrative
    f1  = row.get('forecast_1y',  None)
    f3  = row.get('forecast_3y',  None)
    ft  = row.get('forecast_trend', 'Unknown')
    fm  = row.get('forecast_method', 'Unknown')
    f_text = (
        f"Demand is forecast to reach {round(f1,0)} in 1 year "
        f"and {round(f3,0)} in 3 years ({ft} trend, {fm} model)."
        if f1 is not None else "No forecast data available."
    )

    archetype = row.get('archetype', 'Unknown')
    industries = row.get('industries', [])
    category   = row.get('sub_category', 'Unknown')

    # Historical demand
    hist = demand_series(skill)

    # Compose natural-language narrative
    narrative = (
        f"{tier_narrative} "
        f"It is classified under {category} and belongs to the {archetype} archetype. "
        f"Industries where this skill is relevant: {', '.join(industries) if industries else 'General'}. "
        f"{f_text} "
        f"SES score: {score} ({tier})."
    )

    return {
        'skill'              : skill,
        'sub_category'       : category,
        'ses_score'          : score,
        'ses_tier'           : tier,
        'ses_rank'           : int(row.get('ses_rank', 0)),
        'archetype'          : archetype,
        'industries'         : industries,
        'linkedin_demand'    : round(float(row.get('linkedin_demand', 0) or 0), 4),
        'salary_premium'     : round(float(row.get('salary_premium', 0) or 0), 4),
        'current_usage'      : round(float(row.get('current_usage', 0) or 0), 4),
        'future_interest'    : round(float(row.get('future_interest', 0) or 0), 4),
        'growth_rate'        : round(float(row.get('growth_rate', 0) or 0), 4),
        'global_adoption'    : round(float(row.get('global_adoption_score', 0) or 0), 4),
        'forecast_1y'        : round(float(f1), 2) if f1 is not None else None,
        'forecast_2y'        : round(float(row.get('forecast_2y') or f1 or 0), 2),
        'forecast_3y'        : round(float(f3), 2) if f3 is not None else None,
        'forecast_trend'     : ft,
        'forecast_method'    : fm,
        'forecast_confidence': round(float(row.get('confidence', 0.3)), 2),
        'history_length'     : int(row.get('history_length', 0) or 0),
        'demand_history'     : hist,
        'narrative'          : narrative,
    }

knowledge_base = [build_document(row) for _, row in master.iterrows()]

print(f"Knowledge base entries: {len(knowledge_base)}")
print("\nSample entry:")
print(json.dumps(knowledge_base[0], indent=2)[:1200])


Knowledge base entries: 114

Sample entry:
{
  "skill": "javascript",
  "sub_category": "Programming",
  "ses_score": 0.7676,
  "ses_tier": "Very Safe",
  "ses_rank": 1,
  "archetype": "Future-Proof",
  "industries": [
    "Software Engineering"
  ],
  "linkedin_demand": 0.1043,
  "salary_premium": 0.5038,
  "current_usage": 1.0,
  "future_interest": 0.5832,
  "growth_rate": 0.0728,
  "global_adoption": 1.0,
  "forecast_1y": null,
  "forecast_2y": 0.0,
  "forecast_3y": null,
  "forecast_trend": "Unknown",
  "forecast_method": "Unknown",
  "forecast_confidence": 0.3,
  "history_length": 0,
  "demand_history": {
    "2013": 30526,
    "2014": 75081,
    "2015": 116934,
    "2016": 137039,
    "2017": 160607,
    "2018": 180644,
    "2019": 180763,
    "2020": 203046,
    "2021": 228274,
    "2022": 238526,
    "2023": 244175,
    "2024": 239374,
    "2025": 265328
  },
  "narrative": "javascript is a highly future-proof skill with strong and growing demand. It is classified under Program

In [4]:
# ── save knowledge base JSON ─────────────────────────────────────────────────
kb_path = '../Generated Datasets/skill_knowledge_base.json'
with open(kb_path, 'w') as f:
    json.dump(knowledge_base, f, indent=2)

print(f"Saved → {kb_path}")
print(f"Total documents: {len(knowledge_base)}")


Saved → ../Generated Datasets/skill_knowledge_base.json
Total documents: 114


In [5]:
# ── build FAISS vector index ─────────────────────────────────────────────────
# Uses numeric feature vectors as embeddings.
# In production you would replace this with sentence-transformer embeddings.

try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    FAISS_AVAILABLE = False
    print("faiss-cpu not installed.  Run:  pip install faiss-cpu")
    print("Skipping FAISS index build — knowledge base JSON is still complete.")

if FAISS_AVAILABLE:
    VECTOR_COLS = [
        'linkedin_demand', 'salary_premium', 'current_usage',
        'future_interest', 'growth_rate', 'global_adoption',
        'ses_score',
    ]

    # Build float32 matrix from knowledge base
    vectors = np.array([
        [doc.get(c, 0.0) or 0.0 for c in VECTOR_COLS]
        for doc in knowledge_base
    ], dtype=np.float32)

    # L2-normalise so cosine ~ dot product
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    vectors_norm = vectors / norms

    dim   = vectors_norm.shape[1]
    index = faiss.IndexFlatIP(dim)   # Inner-product (cosine after normalisation)
    index.add(vectors_norm)

    faiss.write_index(index, '../Generated Datasets/faiss_index.bin')
    print(f"FAISS index saved ({index.ntotal} vectors, dim={dim})")

    # Save metadata for lookup
    faiss_meta = [
        {'id': i, 'skill': doc['skill'], 'ses_tier': doc['ses_tier'],
         'archetype': doc['archetype']}
        for i, doc in enumerate(knowledge_base)
    ]
    with open('../Generated Datasets/faiss_metadata.json', 'w') as f:
        json.dump(faiss_meta, f, indent=2)
    print("FAISS metadata saved → faiss_metadata.json")


FAISS index saved (114 vectors, dim=7)
FAISS metadata saved → faiss_metadata.json


In [6]:
# ── verify retrieval ──────────────────────────────────────────────────────────
if FAISS_AVAILABLE:
    # Quick sanity: query for "python"-like vector
    python_doc = next((d for d in knowledge_base if d['skill'] == 'python'), None)
    if python_doc:
        query_vec = np.array(
            [[python_doc.get(c, 0.0) for c in VECTOR_COLS]], dtype=np.float32
        )
        q_norm = query_vec / np.linalg.norm(query_vec)
        D, I   = index.search(q_norm, k=6)

        print("Top 6 skills similar to 'python':")
        for rank, (dist, idx) in enumerate(zip(D[0], I[0]), 1):
            print(f"  {rank}. {faiss_meta[idx]['skill']:30s}  score={dist:.4f}")


Top 6 skills similar to 'python':
  1. python                          score=1.0000
  2. sql                             score=0.9886
  3. docker                          score=0.9842
  4. javascript                      score=0.9824
  5. typescript                      score=0.9802
  6. postgresql                      score=0.9785


In [7]:
# ── summary ───────────────────────────────────────────────────────────────────
print("=" * 50)
print("RAG Knowledge Base — Build Summary")
print("=" * 50)
print(f"  Total skill documents : {len(knowledge_base)}")
print(f"  JSON path             : skill_knowledge_base.json")
if FAISS_AVAILABLE:
    print(f"  FAISS index           : faiss_index.bin  ({dim}D vectors)")
    print(f"  FAISS metadata        : faiss_metadata.json")
print("\nTier breakdown:")
from collections import Counter
tier_dist = Counter(d['ses_tier'] for d in knowledge_base)
for tier in ['Very Safe','Safe','Moderate','Risky','Extinction Risk']:
    print(f"  {tier:20s}: {tier_dist.get(tier, 0)}")


RAG Knowledge Base — Build Summary
  Total skill documents : 114
  JSON path             : skill_knowledge_base.json
  FAISS index           : faiss_index.bin  (7D vectors)
  FAISS metadata        : faiss_metadata.json

Tier breakdown:
  Very Safe           : 3
  Safe                : 78
  Moderate            : 32
  Risky               : 1
  Extinction Risk     : 0
